# LLM Agents & ReAct — PayBharat Case Study

**Audience:** Beginner / intermediate · **LLM:** Ollama `llama3.2:3b`

**Case:** Priya Sharma claims **Rs 1,200** was debited twice on PhonePe — ref **`TXN-UP-7721`**.

**Character:** Aarav Mehta (PayBharat intern) builds support agents step by step.

| Act | Technique |
|-----|-----------|
| 1 | Simple agent (one prompt) |
| 2 | ReAct loop (Thought → Action → Observation) |
| 3 | Plan vs Execute |
| 4 | Decision flow control |
| 5 | Debug wrong reasoning |

> Run cells **top to bottom**. Each cell does **one small job**. Model answers are printed **in full**.


---

## Setup

We load imports and the **same complaint** used in every Act.


In [ ]:
import json
import re

import ollama

MODEL = "llama3.2:3b"
print("Model:", MODEL)


Model: llama3.2:3b


In [ ]:
# ★ Same complaint in every Act ★
TXN_ID = "TXN-UP-7721"

CUSTOMER_MESSAGE = (
    "Hello PayBharat, PhonePe se MSEB bill Rs 1,200 pay kiya kal shaam. "
    "Bank statement mein same amount do baar debit — ref "
    f"{TXN_ID}. Paise wapas chahiye, month end EMI hai. Priya Sharma."
)

print("Ref :", TXN_ID)
print("From:", CUSTOMER_MESSAGE)


Ref : TXN-UP-7721
From: Hello PayBharat, PhonePe se MSEB bill Rs 1,200 pay kiya kal shaam. Bank statement mein same amount do baar debit — ref TXN-UP-7721. Paise wapas chahiye, month end EMI hai. Priya Sharma.


### Simulated tools

Real agents call APIs. We return **fixed JSON** so Observations stay stable in class.


In [ ]:
def run_tool(action: str) -> str:
    action = action.strip()
    if action.upper().startswith("FINISH"):
        return "FINISH"
    if "lookup_transaction" in action:
        return json.dumps({"amount_inr": 1200, "status": "SUCCESS", "merchant": "MSEB Bill Pay"})
    if "check_duplicate" in action:
        return json.dumps({"is_duplicate": True, "original_txn": "TXN-UP-7709"})
    if "get_refund_policy" in action:
        return json.dumps({"eligible": True, "timeline": "5-7 working days"})
    if "calculate_refund" in action:
        return json.dumps({"refund_amount_inr": 1200})
    if "initiate_refund" in action:
        return json.dumps({"refund_id": "REF-9912", "eta": "5-7 working days"})
    return json.dumps({"error": "unknown action"})

print("Tools: lookup_transaction | check_duplicate | get_refund_policy |")
print("       calculate_refund | initiate_refund | FINISH")


Tools: lookup_transaction | check_duplicate | get_refund_policy |
       calculate_refund | initiate_refund | FINISH


In [ ]:
def ask(prompt, max_tokens=200):
    r = ollama.generate(
        model=MODEL, prompt=prompt,
        options={"temperature": 0, "num_predict": max_tokens},
    )
    return r.response.strip()

try:
    ok = any(MODEL in m.model for m in ollama.list().models)
    print("Ollama ✓" if ok else f"⚠ run: ollama pull {MODEL}")
except Exception as e:
    print("Ollama not running —", e)


Ollama ✓


### ReAct helpers

Each step prints the **full Thought**, **full Action**, and **full Observation**.


In [ ]:
def parse_react(text):
    t = re.search(r"Thought\s*:\s*(.+?)(?=\nAction|$)", text, re.S | re.I)
    a = re.search(r"Action\s*:\s*(.+?)(?=\n|$)", text, re.S | re.I)
    return (t.group(1).strip() if t else text.strip(), a.group(1).strip() if a else "")


def format_obs(obs):
    if obs == "FINISH":
        return "✓ agent finished"
    try:
        return json.dumps(json.loads(obs), indent=2, ensure_ascii=False)
    except Exception:
        return obs


def print_step(n, thought, action, obs):
    print(f"Step {n}")
    print(f"  Thought → {thought}")
    print(f"  Action  → {action}")
    print(f"  Obs     →")
    print(format_obs(obs))

print("Helpers ready")


Helpers ready


In [ ]:
REACT_RULES = """You are Aarav (PayBharat support). One step only.

Tools:
- lookup_transaction(upi_ref="TXN-UP-7721")
- check_duplicate(txn_id="TXN-UP-7721")
- get_refund_policy(category="duplicate_debit")
- calculate_refund(txn_id="TXN-UP-7721", reason="duplicate_debit")
- initiate_refund(txn_id="TXN-UP-7721", amount=1200)
- FINISH(summary="1-2 sentence reply to customer")

Format:
Thought: ...
Action: ...

Rules: use Observations — do not invent data. check_duplicate BEFORE initiate_refund.
"""
print("ReAct rules loaded ✓ (6 tools + FINISH)")


ReAct rules loaded ✓ (6 tools + FINISH)


In [ ]:
def history_text(steps):
    if not steps:
        return "(none)"
    lines = []
    for i, s in enumerate(steps, 1):
        lines.append(f"{i}. Action: {s['action']}")
        lines.append(f"   Obs: {format_obs(s['observation'])}")
    return "\n".join(lines)


def react_one_step(complaint, steps, step_num, guard_refund=False, verified=None):
    if verified is None:
        verified = {"ok": False}
    prompt = f"""{REACT_RULES}

Complaint ({TXN_ID}): {complaint}

Done so far:
{history_text(steps)}

Step {step_num} — Thought + Action only."""
    raw = ask(prompt, max_tokens=180)
    thought, action = parse_react(raw)
    if not action:
        return {"thought": thought, "action": "?", "observation": '{"error":"no Action parsed"}'}, False, None
    if action.upper().startswith("FINISH"):
        m = re.search(r"FINISH\((.*)\)", action, re.S | re.I)
        summary = m.group(1).strip() if m else action
        return {"thought": thought, "action": action, "observation": "FINISH"}, True, summary
    if guard_refund and "initiate_refund" in action and not verified["ok"]:
        obs = json.dumps({"error": "BLOCKED — run check_duplicate first"})
        return {"thought": thought, "action": action, "observation": obs}, False, None
    obs = run_tool(action)
    if "check_duplicate" in action and "true" in obs.lower():
        verified["ok"] = True
    return {"thought": thought, "action": action, "observation": obs}, False, None

print("react_one_step() ready ✓")


react_one_step() ready ✓


---

## Act 1 — Simple agent (no tools)

Aarav sends **one prompt**: *"resolve everything now"*.  
No lookups. No Observations.


In [ ]:
simple_prompt = f"""PayBharat support — resolve in ONE reply.
Include: txn status, duplicate?, refund amount, timeline, refund initiated?

Ref: {TXN_ID}
Message: {CUSTOMER_MESSAGE}"""

simple_answer = ask(simple_prompt, max_tokens=400)
print("SIMPLE AGENT — full model answer:")
print("=" * 72)
print(simple_answer)
print("=" * 72)


SIMPLE AGENT — full model answer:
Dear Priya Sharma,

Thank you for reaching out to us regarding the issue with your PayBharat transaction. We apologize for the inconvenience caused.

After verifying your details, we found that there was a duplicate transaction of Rs 1,200 from PhonePe to MSEB bill, which is reflected in your bank statement. This means that the amount has been deducted twice, and you are eligible for a refund.

We have initiated the refund process for the excess amount of Rs 1,200 (Rs 600) with TXN-UP-7721. The refund amount will be credited back to your PayBharat wallet within the next 3-5 business days, which is in line with our standard refund timeline.

Please note that this transaction was processed on the same day as your EMI payment, and we understand that this may have caused some confusion. We assure you that we are taking immediate action to rectify the situation and ensure that you receive your rightful refund amount.

If you have any further questions or co

In [ ]:
text = simple_answer.lower()

dup_ok = any(w in text for w in ["duplicate", "double", "do baar", "7709"])
amt_ok = "1200" in text.replace(",", "")
time_ok = any(w in text for w in ["5-7", "working day"])
bad_init = any(w in text for w in ["refund initiated", "ref-9912", "processed your refund"])

trust = sum([dup_ok, amt_ok, time_ok, not bad_init])
print("TRUST SCORECARD — TXN-UP-7721")
print(f"  {'✓' if dup_ok else '✗'}  mentions duplicate")
print(f"  {'✓' if amt_ok else '✗'}  amount Rs 1,200")
print(f"  {'✓' if time_ok else '✗'}  timeline 5-7 days")
print(f"  {'✗' if bad_init else '✓'}  did NOT fake 'refund initiated'")
print(f"\nScore: {trust}/4  → ops cannot audit a one-shot guess")


TRUST SCORECARD — TXN-UP-7721
  ✓  mentions duplicate
  ✓  amount Rs 1,200
  ✗  timeline 5-7 days
  ✗  did NOT fake 'refund initiated'

Score: 2/4  → ops cannot audit a one-shot guess


**Lesson:** Looks helpful, but nobody verified the transaction. Next: **ReAct** — one step at a time.


---

## Act 2 — ReAct: one step per cell

Pattern: **Thought → Action → Observation** → repeat.

We run **one step per cell** so you see the loop breathe.


In [ ]:
# Step 1 — usually lookup first
react_steps = []
verified = {"ok": False}

s, done, fin = react_one_step(CUSTOMER_MESSAGE, react_steps, 1, verified=verified)
react_steps.append(s)
print_step(1, s["thought"], s["action"], s["observation"])


Step 1
  Thought → The customer has mentioned that there are duplicate debits of Rs 1200 from PhonePe to MSEB bill, which indicates a potential issue with the transaction.
  Action  → check_duplicate(txn_id="TXN-UP-7721")
  Obs     →
{
  "is_duplicate": true,
  "original_txn": "TXN-UP-7709"
}


In [ ]:
# Step 2
s, done, fin = react_one_step(CUSTOMER_MESSAGE, react_steps, 2, verified=verified)
react_steps.append(s)
print_step(2, s["thought"], s["action"], s["observation"])


Step 2
  Thought → Since the transaction is a duplicate debit, we need to initiate a refund.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


In [ ]:
# Step 3
s, done, fin = react_one_step(CUSTOMER_MESSAGE, react_steps, 3, verified=verified)
react_steps.append(s)
print_step(3, s["thought"], s["action"], s["observation"])


Step 3
  Thought → Since the transaction is marked as a duplicate, we need to initiate a refund for the original amount.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


In [ ]:
# Step 4
s, done, fin = react_one_step(CUSTOMER_MESSAGE, react_steps, 4, verified=verified)
react_steps.append(s)
print_step(4, s["thought"], s["action"], s["observation"])


Step 4
  Thought → Since the transaction is marked as a duplicate, we need to initiate refund for the original transaction TXN-UP-7709 instead of TXN-UP-7721.
  Action  → initiate_refund(txn_id="TXN-UP-7709", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


In [ ]:
# Step 5 (stop early if FINISH already reached)
if not done:
    s, done, fin = react_one_step(CUSTOMER_MESSAGE, react_steps, 5, verified=verified)
    react_steps.append(s)
    print_step(5, s["thought"], s["action"], s["observation"])
else:
    print("Agent already FINISH at step 4 ✓")
    fin = fin


Step 5
  Thought → Since the transaction is marked as a duplicate debit, we need to initiate refund for both transactions.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200) and initiate_refund(txn_id="TXN-UP-7709", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


In [ ]:
actions = " ".join(x["action"] for x in react_steps).lower()
print("REACT SCORECARD")
print(f"  {'✓' if 'lookup' in actions else '✗'}  lookup_transaction")
print(f"  {'✓' if 'check_duplicate' in actions else '✗'}  check_duplicate")
print(f"  {'✓' if done else '✗'}  reached FINISH")
print(f"  Steps: {len(react_steps)}")
if fin:
    print("  FINISH summary:")
    print(fin)


REACT SCORECARD
  ✗  lookup_transaction
  ✓  check_duplicate
  ✗  reached FINISH
  Steps: 5


**Pattern:** `while not done: think → act → observe`

Each Observation feeds the **next** Thought — that is grounding.


---

## Act 3 — Plan first, execute later

| Phase | Tools? | Purpose |
|-------|--------|---------|
| **PLAN** | No | Ops lead reviews steps |
| **EXEC** | Yes | Runtime runs the plan |


In [ ]:
plan_prompt = f"""Plan only (no tools) for {TXN_ID}.
Customer: {CUSTOMER_MESSAGE}

List 4-5 numbered steps. Name which tool each step will use.
Format: PLAN:\n1. ...\n2. ..."""

plan_text = ask(plan_prompt, max_tokens=350)
print("PLAN (full — review before any refund):")
print("=" * 72)
print(plan_text)
print("=" * 72)


PLAN (full — review before any refund):
PLAN:

1. **Verify transaction details**: Use PhonePe app to check the transaction history and confirm the amount paid for MSEB bill, TXN-UP-7721.

2. **Check bank statement**: Log in to online banking or mobile banking app of the bank account used for payment to review the debit transactions and verify if there are two debits of Rs 1200 each.

3. **Contact PhonePe customer support**: Reach out to PhonePe's customer support via phone, email, or live chat to report the issue and request assistance in reversing the transaction.

4. **Request reversal from MSEB**: Contact MSEB's customer service to inform them about the incorrect debit of Rs 1200 and request them to reverse the amount back into the account.

5. **Follow up with bank and PhonePe**: Use phone or email to follow up with both the bank and PhonePe to ensure that the transaction is reversed and the amount is credited back into the account before the month-end EMI.


### Execute the plan

We re-run ReAct with the plan in context — **full Action + Observation** each step.


In [ ]:
exec_ctx = f"Approved plan:\n{plan_text}\n\n{CUSTOMER_MESSAGE}"
exec_steps, verified = [], {"ok": False}
exec_final, finished = None, False

for n in range(1, 6):
    if finished:
        break
    s, finished, exec_final = react_one_step(exec_ctx, exec_steps, n, verified=verified)
    exec_steps.append(s)
    print_step(n, s["thought"], s["action"], s["observation"])

print("\nPLAN vs EXEC:")
print("  PLAN = human-readable checklist")
print("  EXEC = what actually ran (compare tool order)")
if exec_final:
    print("\nFINISH summary:")
    print(exec_final)


Step 1
  Thought → The customer has reported a duplicate debit of Rs 1200 for the MSEB bill TXN-UP-7721 and is requesting assistance in reversing the transaction.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


Step 2
  Thought → Before initiating a refund, we need to check if the transaction is a duplicate debit.
  Action  → check_duplicate(txn_id="TXN-UP-7721")
  Obs     →
{
  "is_duplicate": true,
  "original_txn": "TXN-UP-7709"
}


Step 3
  Thought → Since the transaction is marked as a duplicate, we need to initiate a refund and also inform MSEB about the incorrect debit.
  Action  → get_refund_policy(category="duplicate_debit")
  Obs     →
{
  "eligible": true,
  "timeline": "5-7 working days"
}


Step 4
  Thought → Since the transaction is a duplicate debit, we need to initiate a refund and also check with MSEB if they can reverse the amount back into the account.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


Step 5
  Thought → Before initiating the refund, we need to confirm with PhonePe and MSEB that there are indeed two debits of Rs 1200 each.
  Action  → check_duplicate(txn_id="TXN-UP-7721")
  Obs     →
{
  "is_duplicate": true,
  "original_txn": "TXN-UP-7709"
}

PLAN vs EXEC:
  PLAN = human-readable checklist
  EXEC = what actually ran (compare tool order)


---

## Act 4 — Control the decision flow

Three guardrails: **max_steps**, **tool order**, **FINISH**.


In [ ]:
# Guardrail 1 — max_steps=2 (agent may not finish)
mini = []
for n in range(1, 3):
    s, d, _ = react_one_step(CUSTOMER_MESSAGE, mini, n)
    mini.append(s)
    print_step(n, s["thought"], s["action"], s["observation"])
print("\nmax_steps=2 →", len(mini), "steps, FINISH reached:", d)


Step 1
  Thought → The customer has mentioned that there are duplicate debits of Rs 1200 from PhonePe to MSEB bill, which indicates a potential issue with the transaction.
  Action  → check_duplicate(txn_id="TXN-UP-7721")
  Obs     →
{
  "is_duplicate": true,
  "original_txn": "TXN-UP-7709"
}


Step 2
  Thought → Since the transaction is a duplicate debit, we need to initiate a refund.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}

max_steps=2 → 2 steps, FINISH reached: False


In [ ]:
# Guardrail 2 — refund before verify = BLOCKED
bad_action = 'initiate_refund(txn_id="TXN-UP-7721", amount=1200)'
bad_obs = run_tool(bad_action) if verified["ok"] else json.dumps(
    {"error": "BLOCKED — check_duplicate first"}
)
print("Wrong order demo:")
print("  Action →", bad_action)
print("  Obs    →")
print(format_obs(bad_obs))


Wrong order demo:
  Action → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs    →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}


In [ ]:
# Guardrail 3 — run with guard on; count blocks
guarded, verified_g = [], {"ok": False}
blocks = 0
for n in range(1, 7):
    s, done_g, fin_g = react_one_step(
        CUSTOMER_MESSAGE, guarded, n, guard_refund=True, verified=verified_g
    )
    guarded.append(s)
    if "BLOCKED" in s["observation"]:
        blocks += 1
    if done_g:
        break

print(f"Guarded run: {len(guarded)} steps, {blocks} blocked attempt(s)")
for i, s in enumerate(guarded, 1):
    print_step(i, s["thought"], s["action"], s["observation"])
if fin_g:
    print("\nFINISH summary:")
    print(fin_g)


Guarded run: 6 steps, 0 blocked attempt(s)
Step 1
  Thought → The customer has mentioned that there are duplicate debits of Rs 1200 from PhonePe to MSEB bill, which indicates a potential issue with the transaction.
  Action  → check_duplicate(txn_id="TXN-UP-7721")
  Obs     →
{
  "is_duplicate": true,
  "original_txn": "TXN-UP-7709"
}
Step 2
  Thought → Since the transaction is a duplicate debit, we need to initiate a refund.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}
Step 3
  Thought → Since the transaction is marked as a duplicate, we need to initiate a refund for the original amount.
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     →
{
  "refund_id": "REF-9912",
  "eta": "5-7 working days"
}
Step 4
  Thought → Since the transaction is marked as a duplicate, we need to initiate refund for the original transaction TXN-UP-7709 instead of TXN-UP-7721.
  Action  → initiate_r

**Takeaway:** Prompts guide behaviour; **code guardrails** stop costly mistakes.


---

## Act 5 — Debug a wrong reasoning path

Teammate skipped verification. We inspect **step 1 only**, then compare tool order.


In [ ]:
print("✗ BAD step 1 (deterministic teaching example)")
print("  Thought → Customer said duplicate — refund now")
print('  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)')
print("  Obs     → ✗ BLOCKED — check_duplicate first")
print("\nFix: read Observation → change next Action → re-run")


✗ BAD step 1 (deterministic teaching example)
  Thought → Customer said duplicate — refund now
  Action  → initiate_refund(txn_id="TXN-UP-7721", amount=1200)
  Obs     → ✗ BLOCKED — check_duplicate first

Fix: read Observation → change next Action → re-run


In [ ]:
bad_order = ["initiate_refund", "lookup_transaction", "check_duplicate", "initiate_refund", "FINISH"]
good_order = [x["action"].split("(")[0] for x in guarded]

print("ACTION ORDER DIFF")
print("  Bad :", " → ".join(bad_order))
print("  Good:", " → ".join(good_order))

bad_refund_at = bad_order.index("initiate_refund") + 1
good_dup_at = next((i + 1 for i, a in enumerate(good_order) if "check_duplicate" in a), "?")
good_refund_at = next((i + 1 for i, a in enumerate(good_order) if "initiate_refund" in a), "?")

print(f"\nBad : refund at step {bad_refund_at} (too early)")
print(f"Good: check_duplicate at step {good_dup_at}, refund at step {good_refund_at}")


ACTION ORDER DIFF
  Bad : initiate_refund → lookup_transaction → check_duplicate → initiate_refund → FINISH
  Good: check_duplicate → initiate_refund → initiate_refund → initiate_refund → initiate_refund → initiate_refund

Bad : refund at step 1 (too early)
Good: check_duplicate at step 1, refund at step 2


**Debug checklist**

1. Is Observation an error or success JSON?  
2. Which Action broke policy order?  
3. Did Thought ignore prior Observations?  
4. Fix prompt **and** add runtime guardrail.  
5. Diff Action sequences before/after.


---

## Journey — TXN-UP-7721

| Act | Technique | Fix |
|-----|-----------|-----|
| 1 | Simple agent | Baseline — unverified guess |
| 2 | ReAct | Thought → Action → Observation |
| 3 | Plan + Execute | Review plan, then run tools |
| 4 | Flow control | max_steps + BLOCKED guardrail |
| 5 | Debug | Wrong order → diff → fix |

**Regenerate:** `python build_and_run_agent_react_demo.py`

**Try:** Change `max_steps` to 2 · Remove duplicate-before-refund rule · Add `notify_customer` tool to PLAN.
